# Project FORESIGHT — Phase 2A: Exploratory Data Analysis & Demand Characterization

**Objective**: Comprehensive exploratory analysis of the clean, analysis-ready demand universe (`data/processed/analysis_ready.parquet`).

**Strict Boundary**: Analysis and diagnostics ONLY. No model training, no predictive feature engineering.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.config import CFG, PATHS
from src.eda import (
    profile_dataset,
    analyze_demand_distribution,
    analyze_sku_demand_and_intermittency,
    analyze_temporal_patterns,
    investigate_weekly_aggregation,
    analyze_categories,
    analyze_promotions,
    analyze_price_demand,
    investigate_outliers,
    analyze_inventory_coverage,
    audit_feature_availability,
    run_eda,
)

print('Environment initialized successfully. Random seed:', CFG.random_seed)


## 1. Load Analysis-Ready Dataset & Profile


In [2]:
df = pd.read_parquet(PATHS.processed_dir / 'analysis_ready.parquet')
prof = profile_dataset(df)
print(f"Rows: {prof['rows']:,} | Columns: {prof['columns']} | SKUs: {prof['unique_skus']} | Dates: {prof['unique_dates']}")
print(f"Date range: {prof['date_min']} to {prof['date_max']}")
print(f"Complete 50x731 panel: {prof['is_complete_panel']} | Duplicate pairs: {prof['duplicate_grain_count']}")


## 2. Demand Distribution & Statistics


In [3]:
stats = analyze_demand_distribution(df)
for k, v in stats.items():
    print(f"{k:<20}: {v}")


## 3. SKU-Level Demand & Intermittency (Syntetos-Boylan)


In [4]:
sku_df = analyze_sku_demand_and_intermittency(df)
print('Intermittency Classifications:')
print(sku_df['Intermittency_Class'].value_counts())
display_cols = ['SKU', 'Product_Name', 'Category', 'Total_Units_Sold', 'Mean_Daily_Demand', 'CV', 'ADI', 'CV2', 'Intermittency_Class']
print('\nTop 5 SKUs by Volume:')
print(sku_df[display_cols].head())


## 4. Promotional Responsiveness & Lift


In [5]:
promo_df = analyze_promotions(df)
print(promo_df[promo_df['Scope'] == 'Global'])


## 5. Weekly Aggregation vs Daily Signal-to-Noise Ratio


In [6]:
weekly_comp, weekly_df = investigate_weekly_aggregation(df)
for k, v in weekly_comp.items():
    print(f"{k:<25}: {v}")


## 6. Data Leakage & Feature Availability Audit


In [7]:
leakage_df = audit_feature_availability(df)
print(leakage_df[['column', 'source', 'available_at_forecast_time?', 'leakage_risk']].head(10))


## 7. Run Full Pipeline & Verify Artifacts


In [8]:
res = run_eda()
print('EDA executed successfully. Generated plots:')
for name, p in res['plot_paths'].items():
    print(f'  {name:<32}: {p.name}')
